In [67]:
### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [68]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [69]:
### Read all pdf inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in the directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDF in the data directory

all_pdf_documents = process_all_pdfs("../data")
all_pdf_documents

Found 4 PDF files to process

Processing: AD 1.1.pdf
 Loaded 2 pages

Processing: AD 1.2.pdf
 Loaded 6 pages

Processing: AD 1.3.pdf
 Loaded 6 pages

Processing: AD 1.4.pdf
 Loaded 5 pages

Total documents loaded: 19


[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'FrameMaker 2017.0.3', 'creationdate': '2000-08-03T09:29:00+00:00', 'source': '..\\data\\PDF\\AD 1.1.pdf', 'file_path': '..\\data\\PDF\\AD 1.1.pdf', 'total_pages': 2, 'format': 'PDF 1.6', 'title': 'untitled', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-07-29T19:44:58+05:30', 'trapped': '', 'modDate': "D:20210729194458+05'30'", 'creationDate': 'D:20000803092900Z', 'page': 0, 'source_file': 'AD 1.1.pdf', 'file_type': 'pdf'}, page_content='Airports Authority of India\nAIRAC effective date\nIndia\nAIP\nAD 1 AERODROMES/HELIPORTS – INTRODUCTION\nAD 1.1 AERODROME/HELIPORT AVAILABILITY AND CONDITIONS OF USE\n1.1\nCIVIL AERODROMES:\nAll civil aerodromes under the management of Airports Authority of India are open for public use within\npublished operational hours. Current NOTAM may be consulted for latest operational hours.\n1.2\nMILITARY AERODROMES:\nMilitary aerodromes are not available for publ

In [70]:
### Text splitting into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}....")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [71]:
chunks = split_documents(all_pdf_documents)
chunks

Split 19 documents into 50 chunks

Example chunk:
Content: Airports Authority of India
AIRAC effective date
India
AIP
AD 1 AERODROMES/HELIPORTS – INTRODUCTION
AD 1.1 AERODROME/HELIPORT AVAILABILITY AND CONDITIONS OF USE
1.1
CIVIL AERODROMES:
All civil aerodro....
Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'FrameMaker 2017.0.3', 'creationdate': '2000-08-03T09:29:00+00:00', 'source': '..\\data\\PDF\\AD 1.1.pdf', 'file_path': '..\\data\\PDF\\AD 1.1.pdf', 'total_pages': 2, 'format': 'PDF 1.6', 'title': 'untitled', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-07-29T19:44:58+05:30', 'trapped': '', 'modDate': "D:20210729194458+05'30'", 'creationDate': 'D:20000803092900Z', 'page': 0, 'source_file': 'AD 1.1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'FrameMaker 2017.0.3', 'creationdate': '2000-08-03T09:29:00+00:00', 'source': '..\\data\\PDF\\AD 1.1.pdf', 'file_path': '..\\data\\PDF\\AD 1.1.pdf', 'total_pages': 2, 'format': 'PDF 1.6', 'title': 'untitled', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-07-29T19:44:58+05:30', 'trapped': '', 'modDate': "D:20210729194458+05'30'", 'creationDate': 'D:20000803092900Z', 'page': 0, 'source_file': 'AD 1.1.pdf', 'file_type': 'pdf'}, page_content='Airports Authority of India\nAIRAC effective date\nIndia\nAIP\nAD 1 AERODROMES/HELIPORTS – INTRODUCTION\nAD 1.1 AERODROME/HELIPORT AVAILABILITY AND CONDITIONS OF USE\n1.1\nCIVIL AERODROMES:\nAll civil aerodromes under the management of Airports Authority of India are open for public use within\npublished operational hours. Current NOTAM may be consulted for latest operational hours.\n1.2\nMILITARY AERODROMES:\nMilitary aerodromes are not available for publ

### Embedding and VectorStoreDB

In [72]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [73]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Inititalize the embedding manager

        Args: 
        model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model  = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    ## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9345.68it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [74]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 100


In [75]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'FrameMaker 2017.0.3', 'creationdate': '2000-08-03T09:29:00+00:00', 'source': '..\\data\\PDF\\AD 1.1.pdf', 'file_path': '..\\data\\PDF\\AD 1.1.pdf', 'total_pages': 2, 'format': 'PDF 1.6', 'title': 'untitled', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-07-29T19:44:58+05:30', 'trapped': '', 'modDate': "D:20210729194458+05'30'", 'creationDate': 'D:20000803092900Z', 'page': 0, 'source_file': 'AD 1.1.pdf', 'file_type': 'pdf'}, page_content='Airports Authority of India\nAIRAC effective date\nIndia\nAIP\nAD 1 AERODROMES/HELIPORTS – INTRODUCTION\nAD 1.1 AERODROME/HELIPORT AVAILABILITY AND CONDITIONS OF USE\n1.1\nCIVIL AERODROMES:\nAll civil aerodromes under the management of Airports Authority of India are open for public use within\npublished operational hours. Current NOTAM may be consulted for latest operational hours.\n1.2\nMILITARY AERODROMES:\nMilitary aerodromes are not available for publ

In [76]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 50 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.38it/s]

Generated embeddings with shape: (50, 384)
Adding 50 documents to vector store...
Successfully added 50 documents to vector store
Total documents in collection: 150


In [77]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [78]:
rag_retriever.retrieve("What is the scope of Rescue and Fire Service")

Retrieving documents for query: 'What is the scope of Rescue and Fire Service'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 126.14it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8087e04f_6',
  'content': 'greatest opportunities of saving lives.\n1.1.7\nThe extent of aircraft fires which may affect the rescue is influence largely by the quantity and deposition of fuel\ncarried by the aircraft and the location of any fuel release as the result of the accident or incident.\n1.1.8\nThe proposal set out here under concerning these services are intended as a general guide, to be applied to the\nfullest extent practical.\n1.1.9\nWhere an aerodrome is located close to water/sea/swamp areas, or difficult terrain/environment and where a\nsignificant portion of approach or departure operations takes place over these areas, specialist rescue services and\nfirefighting equipment/appliances appropriate to the hazard and risk shall be available.\n1.2\nADMINISTRATION\n1.2.1 \nThe Rescue & Fire Service at an airport should be under the administrative control of Airports Authority of India.\n1.2.2 \nThe Rescue & Fire Service should be responsible for ensuring that 

In [79]:
rag_retriever.retrieve("Domestic Airport/Civil Enclave")

Retrieving documents for query: 'Domestic Airport/Civil Enclave'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 142.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_705d56f7_39',
  'content': 'Airports Authority of India\nIndia\nAIP\nAD 1.4 GROUPING OF AERODROMES AND HELIPORTS\n1.\nType of Airport\nINTL - International Airport/Civil Enclave\nAn airport/aerodrome (including those military airports/aerodromes where civil air traffic is allowed\nunder certain conditions) of entry and departure for international air traffic, where all formalities\nconcerning customs, immigration, health, animal and plant quarantine and similar procedures are\ncarried out and where air traffic services are available on a regular basis.\nCUST - Custom Airport/Civil Enclave\nAn airport/aerodrome (including those military airports/aerodromes where civil air traffic is allowed\nunder certain conditions) available for the entry or departure for international air traffic, where the\nformalities concerning customs, immigration, health and similar procedures and air traffic services are\nmade available, on a restricted basis, to flights with prior approval only.\n

In [80]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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


In [93]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [92]:
answer=rag_simple("What is Custom Airport/Civil Enclave",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Custom Airport/Civil Enclave'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 121.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

 Airports Authority of India
India
AIP
AD 1.4 GROUPING OF AERODROMES AND HELIPORTS
1.
Type of Airport
INTL - International Airport/Civil Enclave
An airport/aerodrome (including those military airports/aerodromes where civil air traffic is allowed
under certain conditions) of entry and departure for international air traffic, where all formalities
concerning customs, immigration, health, animal and plant quarantine and similar procedures are
carried out and where air traffic services are available on a regular basis.
CUST - Custom Airport/Civil Enclave
An airport/aerodrome (including those military airports/aerodromes where civil air traffic is allowed
under certain conditions) available for the entry or departure for international air traffic, where the
formalities concerning customs, immigration, health and similar procedures and air traffic services are
made available, on a restricted basis, to flights

Custom Airport/Civil Enclave (CUST) is an airport/aerodrome available for international air traffic entry or departure with restricted formalities (customs, immigration, health) and air traffic services, available only to flights with prior approval.


In [97]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Location Indicator of JABALPUR Airport", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Location Indicator of JABALPUR Airport'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 93.35it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: VAJB
Sources: [{'source': 'AD 1.4.pdf', 'page': 2, 'score': 0.13906991481781006, 'preview': 'AIRPORT , NAGPUR\nVANP\nINTERNATIONAL\nDURGAPUR AIRPORT\nVEDG\nDOMESTIC\nGAYA AIRPORT\nVEGY\nCUSTOM\nGOA INTERNATIONAL AIRPORT,DABOLIM,GOA \n(NAVY)\nVOGO\nCIVIL ENCLAVE-\nINTERNATIONAL\nGONDIA AIRPORT\nVAGD\nDOMESTIC\nGORAKHPUR AIRPORT\nVEGK\nCIVIL ENCLAVE\nGUWAHATI AIRPORT\nVEGT\nINTERNATIONAL\nGWALIOR AIRPORT\nVIGR\nCIVIL...'}, {'source': 'AD 1.4.pdf', 'page': 2, 'score': 0.13906991481781006, 'preview': 'AIRPORT , NAGPUR\nVANP\nINTERNATIONAL\nDURGAPUR AIRPORT\nVEDG\nDOMESTIC\nGAYA AIRPORT\nVEGY\nCUSTOM\nGOA INTERNATIONAL AIRPORT,DABOLIM,GOA \n(NAVY)\nVOGO\nCIVIL ENCLAVE-\nINTERNATIONAL\nGONDIA AIRPORT\nVAGD\nDOMESTIC\nGORAKHPUR AIRPORT\nVEGK\nCIVIL ENCLAVE\nGUWAHATI AIRPORT\nVEGT\nINTERNATIONAL\nGWALIOR AIRPORT\nVIGR\nCIVIL...'}]
Confidence: 0.13906991481781006
Context Preview: AIRPORT , NAGPUR
VANP
INTERNATIONAL
DURGAPUR AIRPORT
VEDG
DOMESTIC
GAYA AIRPORT
VEGY
CUSTOM
GOA INTERNATIONA

In [100]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Type of airport", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Type of airport'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 140.76it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Airports Authority of India
India
AIP
AD 1.4 GROUPING OF AERODROMES AND HELIPORTS
1.
Type of Airport
INTL - International Airport/Civil Enclave
An airport/aerodrome (including those military airports/aerodromes where civil air traffic is allowed
under

 certain conditions) of entry and departure for international air traffic, where all formalities
concerning customs, immigration, health, animal and plant quarantine and similar procedures are
carried out and where air traffic services are available on a regular basis.
CUST - Custom Airport/Civil Enclave
An airport/aerodrome (including those military airports/aerodromes where civil air traffic is allowed
under certain conditions) available for the entry or departure for international air traffic, where the
formalities concerning customs, immigration, health and similar procedures and air traffic services are
made available, on a restricted basis, to flights with prior approval only.
DOM - Domestic Airport/Civil Enclave

Airports Authority of India
India
AIP
AD 1.4 GROUPING OF AERODROMES AND HELIPORTS
1.
Type of Airport
INTL - International Airport/Civil Enclave
An airport/aerodrome (including those military airports/aerodromes where civil air traffic is allowed
under certain conditions